# 00 — Start here (Student workbook)

**Time:** 15 minutes
**Goal:** confirm the environment, load reduced SANS data, and recognize the
three-column `Q,I,dI` convention.

Work in pairs. Discuss every prediction before running the next cell.

> **Lecture connection**
>
> This practical picks up exactly where last week's lecture stopped: at a
> reduced, radially averaged curve. Keep the lecture slides open beside you.
>
> | Today's task | Lecture slides |
> |---|---|
> | Convert q to a real-space length | 9 *Scattering geometry*, 10 *Real and Q-space* |
> | Read the shape of a log–log curve | 21 *Guinier analysis*, 22 *Power law slopes* |
> | Know what "reduced data" has already been through | 41 *Q-range and resolution*, 43–46 *Data reduction* … *When data look wrong* |
>
> **Notation.** The lecture writes the scattering vector as $Q$; the code and data
> files write `q` or `Q`. They are the same quantity,
> $Q = 4\pi\sin\theta/\lambda$ with $2\theta$ the scattering angle, here in Å⁻¹.
> Always check θ versus 2θ and Å⁻¹ versus nm⁻¹ when you open someone else's data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
DATA = ROOT / 'data'

# This workshop uses sans-fitter 0.4 or newer (pip install "sans-fitter>=0.4").
print(f'Using sans-fitter {sans_fitter.__version__}')
major, minor = (int(part) for part in sans_fitter.__version__.split('.')[:2])
assert (major, minor) >= (0, 4), 'Please install sans-fitter 0.4 or newer'
%matplotlib inline


In [ ]:
filename = DATA / 'sphere_reference.csv'
values = np.genfromtxt(filename, delimiter=',', names=True)
q, intensity, uncertainty = values['Q'], values['I'], values['dI']
print(f'{len(q)} points; q = {q.min():.4f}–{q.max():.3f} Å⁻¹')
assert np.all(np.diff(q) > 0)
assert np.all(uncertainty > 0)


## From q to real space

The lecture rule of thumb is that a feature of size $d$ shows up near
$Q \approx 2\pi/d$: **big in real space, small in reciprocal space**. The cell
below converts the measured q window into the real-space distances it can see.
Before running it, predict roughly what the largest and smallest distances will be.

In [ ]:
d_max = 2 * np.pi / q.min()
d_min = 2 * np.pi / q.max()
print(f'Largest distance probed:  {d_max:6.0f} Å = {d_max / 10:5.1f} nm')
print(f'Smallest distance probed: {d_min:6.1f} Å = {d_min / 10:5.1f} nm')


## Predict before plotting

1. Why are both axes usually logarithmic for SANS?
2. What information does `dI` carry?
3. Which part of the curve is most sensitive to overall particle size?

Write a one-sentence prediction for each in the answer cell below.

*Hints from the lecture:* intensity typically falls by several orders of
magnitude over the measured range, and power laws $I \propto Q^{-m}$ become
straight lines of slope $-m$ on log–log axes (slide 22). Size information sits
where $Q \sim 2\pi/d$ for the whole particle, i.e. at the low-q end (slide 21).

**Your answer** *(double-click this cell to edit it, then press Shift + Enter)*

Write a one-sentence prediction for each of the three questions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(q, intensity, yerr=uncertainty, fmt='o', ms=3, capsize=2)
ax.set(xscale='log', yscale='log', xlabel=r'$q$ / Å$^{-1}$', ylabel=r'$I(q)$')
ax.grid(alpha=0.25)
plt.show()


## Reading the curve

Use the lecture's reading guide on the plot above, moving from left to right:

- **Low-q plateau (Guinier region).** For $Q R_g \lesssim 1.3$ the whole
  particle scatters almost in phase, $I(Q) \approx I(0)\exp(-Q^2R_g^2/3)$, and
  the curve flattens. For a solid sphere $R_g = \sqrt{3/5}\,R$ (slide 21).
- **Oscillations (form factor).** Sharp minima mean nearly identical particles.
  For a sphere the first minimum sits at $Q_1 R \approx 4.49$ (slide 23).
- **High-q decay.** Smooth, sharp interfaces give the Porod envelope
  $I \propto Q^{-4}$ (slide 22).
- **Flat floor at the highest q.** Background, mostly incoherent scattering
  from hydrogen, which is $Q$-independent to a good approximation (slide 15).

Mark each region on a sketch of your plot. Read off $Q_1$ as accurately as you
can and estimate the particle radius. You will need this number as a starting
value in notebook 01.

## Quick check

The curve is already reduced: detector corrections, normalization, radial
averaging, and uncertainty propagation happened upstream. This workshop starts
at the point where a scientific model is compared with those reduced data.

Two consequences matter for the rest of the afternoon:

1. **Units.** In the lecture, $I(Q)$ is an absolute cross-section in cm⁻¹ and
   the master equation (slide 20) lets you check the fitted volume fraction.
   These workshop data are in *arbitrary units*, so the fitted `scale`
   parameter absorbs the volume fraction *and* any calibration factor.
2. **Reduction artefacts look like physics.** Slide 46 (*When data look
   wrong…*) is a useful diagnostic when a residual pattern puzzles you: a
   discrepancy at high q only suggests a flat or weak-signal term (cell,
   background); a constant factor everywhere suggests normalization, thickness,
   transmission or calibration; a change of shape suggests inconsistent
   subtraction. The synthetic data here are clean, but real data rarely are.
3. **Resolution.** Reduced files from a real instrument often carry a fourth
   column, `dQ`, the q resolution of each point (slide 41). sans-fitter (like
   SasView) then smears the model with it, so sharp minima are compared fairly;
   `set_resolution(...)` chooses the mode. The workshop files have no `dQ`
   column, so every notebook calls `set_resolution('none')` to say so.

## Further reading

- [Gommes, Jaksch & Frielinghaus, *Small-angle scattering for beginners*, J. Appl. Cryst. 54, 1832 (2021)](https://pmc.ncbi.nlm.nih.gov/articles/PMC8662971/): an almost mathematics-free introduction that matches the lecture's
  phase-map picture of the form factor.
- [Jacques & Trewhella, *Small-angle scattering for structural biology: expanding the frontier while avoiding the pitfalls*, Protein Sci. 19, 642 (2010)](https://pmc.ncbi.nlm.nih.gov/articles/PMC2867006/): the classic checklist of what can go wrong between sample and curve.
- [SasView SLD calculator](https://www.sasview.org/docs/user/qtgui/Calculators/sld_calculator_help.html): compute the scattering length
  densities that the models in this workshop take as fixed inputs.